In [0]:
from pyspark.sql import Row

docs = [
    Row(id=1, source="email", title="Northern Region May revenue shortfall explanation",
        content="From James Fletcher, 4 June 2026. Northern Region's May revenue came in about 35% below budget for cost centre CC-1010. Root cause: our largest Northern account, Customer-482, pushed their contract renewal from May into July - a timing decision tied to their budget cycle, not a loss. Two mid-size deals forecast to close in May also slipped to early Q3. Impact: about $310k of expected May revenue moved out of the quarter. Cost lines were on plan. Outlook: pipeline intact, Customer-482 renewal now contracted for July, recovery expected in Q3. No structural concern."),
    Row(id=2, source="email", title="May close variance flags",
        content="From Priya Kaur, 2 June 2026. Confirming the only material revenue variance this month is Northern Region. Southern, Eastern, and Western all within tolerance on revenue. Northern's issue is the Customer-482 renewal timing James flagged - purely phasing. Board pack should note this as a timing variance, recoverable in Q3, not a demand problem."),
    Row(id=3, source="sharepoint", title="Budget commentary - Northern Region May",
        content="Business Unit: Northern Region. Cost Centre: CC-1010. Month: 2026-05. Variance Flagged: Revenue -35% vs budget. Commentary: Major client Customer-482 delayed contract renewal from May to July; two large deals slipped out of quarter. Pipeline intact but timing hit May revenue hard. Owner: J. Fletcher. Action: Renewal rescheduled July; forecast recovery Q3."),
    Row(id=4, source="sharepoint", title="Budget commentary - Southern Region March",
        content="Business Unit: Southern Region. Cost Centre: CC-2020. Month: 2026-03. Variance Flagged: Cost +12% vs budget. Commentary: One-off software migration spend, pre-approved by CFO. Non-recurring. Owner: M. Adeyemi. Action: Closed - approved."),
    Row(id=5, source="sharepoint", title="Budget commentary - Western Region June",
        content="Business Unit: Western Region. Cost Centre: CC-4020. Month: 2026-06. Variance Flagged: Cost +9% vs budget. Commentary: Travel costs above plan due to client onsite work; expected to normalise. Owner: T. Nguyen. Action: Monitor."),
]

df = spark.createDataFrame(docs)
(df.write.format("delta")
   .mode("overwrite")
   .option("delta.enableChangeDataFeed", "true")
   .option("overwriteSchema", "true")
   .saveAsTable("finance_documents"))

display(spark.sql("SELECT id, source, title FROM finance_documents"))

id,source,title
1,email,Northern Region May revenue shortfall explanation
2,email,May close variance flags
3,sharepoint,Budget commentary - Northern Region May
4,sharepoint,Budget commentary - Southern Region March
5,sharepoint,Budget commentary - Western Region June


In [0]:
display(spark.sql("""
SELECT * FROM vector_search(
  index => 'workspace.default.document_search',
  query_text => 'Why did Northern Region revenue drop in May?',
  num_results => 3
)
"""))

id,source,title,content,search_score
1,email,Northern Region May revenue shortfall explanation,"From James Fletcher, 4 June 2026. Northern Region's May revenue came in about 35% below budget for cost centre CC-1010. Root cause: our largest Northern account, Customer-482, pushed their contract renewal from May into July - a timing decision tied to their budget cycle, not a loss. Two mid-size deals forecast to close in May also slipped to early Q3. Impact: about $310k of expected May revenue moved out of the quarter. Cost lines were on plan. Outlook: pipeline intact, Customer-482 renewal now contracted for July, recovery expected in Q3. No structural concern.",0.673669
3,sharepoint,Budget commentary - Northern Region May,Business Unit: Northern Region. Cost Centre: CC-1010. Month: 2026-05. Variance Flagged: Revenue -35% vs budget. Commentary: Major client Customer-482 delayed contract renewal from May to July; two large deals slipped out of quarter. Pipeline intact but timing hit May revenue hard. Owner: J. Fletcher. Action: Renewal rescheduled July; forecast recovery Q3.,0.5960134
2,email,May close variance flags,"From Priya Kaur, 2 June 2026. Confirming the only material revenue variance this month is Northern Region. Southern, Eastern, and Western all within tolerance on revenue. Northern's issue is the Customer-482 renewal timing James flagged - purely phasing. Board pack should note this as a timing variance, recoverable in Q3, not a demand problem.",0.5235138
